# CNMP — Carga gold (Resolução 277)

Lê o silver (Warehouse `mp_silver`) via cross-database query e recarrega o
modelo dimensional simplificado no Warehouse `mp_gold`, usando
`python/src/modulos/cnmp/etl/load_gold.py` do repositório
`mpsp-jurimetria/proj202607`.

**Pré-requisitos:**
- Rodar `01_carga_silver.ipynb` antes (o gold lê do silver, não do bronze).
- `mp_silver` e `mp_gold` precisam estar no mesmo workspace Fabric — cross-database
  query só funciona dentro do mesmo workspace.
- Identidade nativa do notebook precisa ter leitura no `mp_silver` e escrita no `mp_gold`.

In [ ]:
%pip install --quiet git+https://github.com/mpsp-jurimetria/proj202607.git#subdirectory=python

## Configuração

Mesma observação do notebook de silver: não são segredos, mas evite deixar
valores reais commitados aqui.

In [ ]:
import os

os.environ["FABRIC_WAREHOUSE_SILVER_NAME"] = "mp_silver"
os.environ["FABRIC_WAREHOUSE_GOLD_HOST"] = "<host>.datawarehouse.fabric.microsoft.com"
os.environ["FABRIC_WAREHOUSE_GOLD_NAME"] = "mp_gold"

In [ ]:
from src.infra.warehouse import get_gold_engine
from src.modulos.cnmp.etl.load_gold import carregar_gold

engine = get_gold_engine()
carregar_gold(engine)

## Verificação

In [ ]:
from sqlalchemy import text

tabelas = [
    "dim_unidade", "dim_formulario", "dim_campo", "dim_campo_opcao",
    "fato_visita", "fato_resposta_tipada",
]

with engine.connect() as conn:
    for tabela in tabelas:
        total = conn.execute(text(f"SELECT COUNT(*) FROM {tabela}")).scalar()
        print(f"{tabela}: {total} linhas")